[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/05_%E5%AE%9F%E8%B7%B5_BtoB%E3%83%9E%E3%83%BC%E3%82%B1/05_%E5%BA%83%E5%91%8A%E3%82%AF%E3%83%AA%E3%83%83%E3%82%AF%E3%81%AE%E4%BA%88%E6%B8%AC.ipynb)

In [ ]:
# === ① セットアップ（最初に実行してください）===
import pandas as pd               # 表データ
import numpy as np                # 数値計算
import matplotlib.pyplot as plt   # グラフ描画
import os
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の文字化け防止
# ローカルに data/ が無ければ公開リポジトリ(raw)から読み込む
RAW = 'https://raw.githubusercontent.com/Carlo-Broschi/statistics-python-for-students/main/data/'
def load(name):
    p = f'../data/{name}'
    return pd.read_csv(p) if os.path.exists(p) else pd.read_csv(RAW + name)
print('準備OK')

# 実践マーケ-05. 広告クリックの予測（相関・単回帰）

**舞台設定**：「広告の**表示(インプレッション)**が増えれば**クリック**も増えるのか？ 表示1000回あたり何クリック取れているのか（≒平均CTR）？」月×チャネルの実績から、表示とクリックの関係を**直線**で定量化します。

**使う統計（読む=3級／本質=2〜3級）**：相関係数・単回帰（最小二乗）・決定係数。

### 使うデータ：`web_marketing.csv`（月×チャネルのマーケ実績30行）
1行＝ある月・あるチャネルの実績。`表示回数` と `クリック数` の関係を見ます。

| 月 | チャネル | 表示回数 | クリック数 | 獲得数 | 費用 |
|---|---|---|---|---|---|
| 2026-01 | 展示会 | 1855 | 149 | 22 | 4363 |

## この章でできるようになること
- 散布図と相関係数で「2つの量の関係」を読める
- 単回帰で「表示が1000回増えるとクリックが何回増えるか（傾き≒CTR）」を求められる
- 決定係数 R² で「直線がどれだけ説明できているか」を判断できる

> **前提**：統計3級（相関・回帰）　/　**所要**：約20分　/　**レベル**：実践（読む3級・本質2〜3級）

## 1. まず散布図で関係を眺める

In [ ]:
mk = load('web_marketing.csv')
fig, ax = plt.subplots(figsize=(7,4.5))
ax.scatter(mk['表示回数'], mk['クリック数'], alpha=.6)
ax.set_xlabel('表示回数'); ax.set_ylabel('クリック数'); ax.set_title('表示回数 vs クリック数'); plt.show()

## 2. 相関係数で「関係の強さ」を数値化

相関係数 r は −1〜+1。+1に近いほど「表示が多いほどクリックも多い」という右肩上がりの関係が強い、を表します。

In [ ]:
r = mk['表示回数'].corr(mk['クリック数'])
print(f'相関係数 r = {r:.3f}')
print('（目安）|r|>0.7 で強い・0.4〜0.7でやや強い・<0.2でほぼ無関係')

## 3. 単回帰：表示回数からクリック数を予測する直線

`クリック数 ≈ 傾き × 表示回数 + 切片` を最小二乗法で求めます。**傾き**は「表示が1回増えるとクリックが何回増えるか」＝平均CTR。1000回あたりに直すと読みやすいです。

In [ ]:
import statsmodels.api as sm
X = sm.add_constant(mk['表示回数']); y = mk['クリック数']
m = sm.OLS(y, X).fit()
slope = m.params['表示回数']; intercept = m.params['const']
print(f'傾き = {slope:.5f} クリック/表示  → 表示1000回あたり 約 {slope*1000:.1f} クリック（≒CTR {slope:.1%}）')
print(f'切片 = {intercept:.1f}')
print(f'決定係数 R² = {m.rsquared:.3f}  （1に近いほど直線でよく説明できる）')

In [ ]:
# 回帰直線を散布図に重ねる
xx = np.linspace(mk['表示回数'].min(), mk['表示回数'].max(), 50)
fig, ax = plt.subplots(figsize=(7,4.5))
ax.scatter(mk['表示回数'], mk['クリック数'], alpha=.6, label='実績')
ax.plot(xx, slope*xx + intercept, color='crimson', label='回帰直線')
ax.set_xlabel('表示回数'); ax.set_ylabel('クリック数'); ax.legend(); ax.set_title('単回帰'); plt.show()

## 4. 予測してみる（と、その注意）

In [ ]:
for imp in [10000, 30000]:
    print(f'表示 {imp:,}回 → 予測クリック数 {slope*imp + intercept:.0f} 回')

> この直線は「表示がクリックを生む」**因果の証明ではありません**。チャネルによって元の反応率が違うなど、別の要因が効いている可能性があります（→ 06章の因果推論へ）。
>
> ちなみに `費用→獲得数` で同じことをすると相関は**マイナス**に出ます。高単価チャネルほど獲得効率が低いためで、これは3級02「シンプソンのパラドックス」と同じ落とし穴です。

## まとめ
- **相関係数**で関係の強さ・向きを、**単回帰**で「1万円あたり何件」という効き目を数値化できる。
- **決定係数 R²** は「直線がデータをどれだけ説明できたか」の割合。
- 予測は便利だが、**観測データの回帰は因果を意味しない**。

> **よくある間違い**
> - **相関≠因果**。関係があっても、片方がもう片方を「起こした」とは限らない。
> - データ範囲の外への **外挿**（例：費用100万円の予測）は危険。
> - R²が高くても、外れ値や非直線の関係を見落としていることがある（必ず散布図を見る）。

## 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます。

In [ ]:
# 採点用の関数（答え合わせに使うだけ・答えは表示しません）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
import numpy as np
# Q1: x=[1,2,3], y=[2,4,6] の相関係数を ans に（完全な直線関係）
ans = None   # 例: np.corrcoef([1,2,3],[2,4,6])[0,1]
_check('Q1 相関係数', ans, np.corrcoef([1,2,3],[2,4,6])[0,1])

In [ ]:
import numpy as np
# Q2: x=[1,2,3], y=[2,4,6] を直線で当てはめたときの傾きを ans に
ans = None   # 例: np.polyfit([1,2,3],[2,4,6],1)[0]
_check('Q2 回帰の傾き', ans, np.polyfit([1,2,3],[2,4,6],1)[0])

In [ ]:
# Q3: 直線がデータを完全に説明できているとき、決定係数 R² はいくつ？ を ans に
ans = None   # ヒント: R²は0〜1。完全な当てはまりなら？
_check('Q3 決定係数', ans, 1.0)

---
## ワークシート課題

**課題1.** `クリック数` から `獲得数` を単回帰してみよう。相関は強い？弱い？ なぜそうなるか考えよう（ヒント：チャネルで獲得効率が大きく違う）。

**課題2.** チャネル（`チャネル`列）で色分けした散布図を描き、チャネルごとに傾きが違いそうか観察しよう。

**課題3.**（考察）R²が同じでも、散布図を見ると当てはめが妥当でない場合がある。どんなときか1〜2行で説明しよう。

> **解答例は別ページ**（ネタバレ防止）**[解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/05_%E5%AE%9F%E8%B7%B5_BtoB%E3%83%9E%E3%83%BC%E3%82%B1/05_%E5%BA%83%E5%91%8A%E3%82%AF%E3%83%AA%E3%83%83%E3%82%AF%E3%81%AE%E4%BA%88%E6%B8%AC.md)**
>
> 自分で解いてから開きましょう。

## 用語集 ＆ チートシート

| 用語 | 意味 |
|---|---|
| 相関係数 r | −1〜+1。関係の強さと向き |
| 単回帰 | y ≈ 傾き×x + 切片（最小二乗） |
| 傾き（回帰係数） | xが1増えるとyが何増えるか |
| 決定係数 R² | 0〜1。直線で説明できた割合 |

```python
r = df['x'].corr(df['y'])
import statsmodels.api as sm
m = sm.OLS(df['y'], sm.add_constant(df['x'])).fit()
m.params; m.rsquared
```

## 完了ログ

このノートを終えたら下のセルを実行すると、学習の完了が記録されます。
**学習者コードは最初に一度だけ設定**すれば、以降は全ノートで自動送信されます（名前の再入力は不要）。

- Colab：左の鍵アイコン（シークレット）で `STUDENT_CODE` に配布コードを登録（1回だけ）
- ローカル：環境変数 `STUDENT_CODE` を設定（または初回に画面入力すると保存され、次回から不要）

In [ ]:
# === 完了ログ ===  学習者コードは最初に一度だけ設定すれば、全ノートで自動送信されます。
import os, datetime, pathlib
try:
    import requests
except ImportError:
    requests = None

def _student_code():
    try:                                          # Colab: 鍵アイコンで STUDENT_CODE を登録
        from google.colab import userdata
        c = userdata.get('STUDENT_CODE')
        if c: return c.strip()
    except Exception:
        pass
    c = os.environ.get('STUDENT_CODE')            # ローカル: 環境変数
    if c: return c.strip()
    p = pathlib.Path.home() / '.student_code'      # 保存ファイル
    if p.exists(): return p.read_text().strip()
    try:                                           # 最後の手段: 入力して保存（次回から不要）
        c = input('学習者コードを入力（配布されたもの）: ').strip()
        if c: p.write_text(c); return c
    except Exception:
        pass
    return ''

CODE = _student_code()
LOG_URL = ""      # 配布時に設定
LOG_TOKEN = ""    # 配布時に設定
NOTEBOOK = "05_実践_BtoBマーケ/05_広告クリックの予測"

if CODE and LOG_URL and requests:
    try:
        requests.post(LOG_URL, json={"token": LOG_TOKEN, "code": CODE, "notebook": NOTEBOOK,
                      "time": datetime.datetime.now().isoformat(timespec="seconds")}, timeout=10)
        print(f"記録しました: {CODE} / {NOTEBOOK}")
    except Exception as e:
        print("記録に失敗しました（URL/ネットワークを確認）:", e)
elif not CODE:
    print("学習者コード未設定。Colabは鍵アイコンで STUDENT_CODE を登録、ローカルは環境変数 STUDENT_CODE を設定してください。")
else:
    print(f"{NOTEBOOK}: LOG_URL/LOG_TOKEN が未設定です（配布時に設定されます）。")